### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import json
import os
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

### Config

In [2]:
CSV_PATH = "prices_expanded.csv"
MODEL_PATH = "price_model.pkl"
ENCODER_DIR = "price_encoders"
os.makedirs(ENCODER_DIR, exist_ok=True)

### Load Data

In [3]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

Loaded 4824 rows
Columns: ['make', 'model', 'year_range', 'part_name', 'part_condition', 'price']


### Encode Categorical Features

In [4]:
cat_cols = ["make", "model", "year_range", "part_name", "part_condition"]
encoders = {}

for col in cat_cols:
  le = LabelEncoder()
  df[col + "_enc"] = le.fit_transform(df[col].astype(str))
  encoders[col] = le
  # Save encoder as JSON for inference
  mapping = {cls: int(i) for i, cls in enumerate(le.classes_)}
  with open(os.path.join(ENCODER_DIR, f"{col}.json"), "w") as f:
    json.dump(mapping, f, indent=2)
  print(f" {col}: {list(le.classes_)}")

 make: ['changan', 'ford', 'volkswagen']
 model: ['estar', 'fusion', 'id4']
 year_range: ['2010_2012', '2013_2016', '2017', '2018_2020', '2020_2026']
 part_name: ['Front-Windscreen-Damage', 'Headlight-Damage', 'Rear-windscreen-Damage', 'Sidemirror-Damage', 'Taillight-Damage', 'bonnet-dent', 'boot-dent', 'doorouter-dent', 'fender-dent', 'front-bumper-dent', 'quaterpanel-dent', 'rear-bumper-dent']
 part_condition: ['aftermarket', 'original_new', 'original_used']


### Features and Target

In [5]:
feature_cols = [col + "_enc" for col in cat_cols]

X = df[feature_cols].values
y = df["price"].values

### Train-Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")


Train: 3859 | Test: 965


### Train Random Forest Regressor

In [7]:
print("\nTraining Random Forest Regressor...")
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)


Training Random Forest Regressor...


RandomForestRegressor(n_jobs=-1, random_state=42)

### Evaluate

In [8]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nTest MAE: ${mae:.2f}")
print(f"Test R2: {r2:.4f}")


Test MAE: $6.92
Test R2: 0.9933


### Feature Importance

In [9]:
print("\nFeature Importances:")
for col, imp in zip(cat_cols, model.feature_importances_):
  print(f"{col}: {imp:.4f}")


Feature Importances:
make: 0.0499
model: 0.0444
year_range: 0.1809
part_name: 0.2828
part_condition: 0.4421


### Save Model

In [10]:
with open(MODEL_PATH, "wb") as f:
  pickle.dump(model, f)

print(f"\nModel saved to {MODEL_PATH}")
print(f"Encoders saved to {ENCODER_DIR}/")


Model saved to price_model.pkl
Encoders saved to price_encoders/
